# Model selection and evaluation

**Real Estate Investment Advisor** | Candidate comparison, cross-validation, and held-out city evaluation


## Purpose and decision boundary

This notebook compares candidate classifiers and regressors, selects each model using cross-validation, and reports performance on cities held out from training. It complements notebook 01 with the model-evaluation stage of the analysis.

**Critical limitation:** the supplied data contains no observed profitability or resale outcomes. Good_Investment is generated from a rule based on property age; Price_After_5Y_Proxy is today's asking price compounded by an assumed rate. The reported scores evaluate reconstruction of those rules and do not establish real investment or future-price accuracy.


### Setup A - Imports


In [ ]:
# All notebook imports are grouped here.
import json
import runpy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")


### Setup B - Project root and output paths


In [ ]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((
    path.resolve() for path in candidate_roots
    if (path / "src" / "config.py").exists() and (path / "data" / "india_housing_prices.csv").exists()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
print(f"Project root: {PROJECT_ROOT}")
print(f"Model outputs: {ARTIFACT_DIR}")


- One reproducible 20% city-group holdout is reserved for final evaluation; its cities are not used for fitting or tuning.
- Candidate-family ranking uses five-fold GroupKFold by City on the training cities; preprocessing is refitted inside each fold.
- After selecting the family, a small predeclared parameter grid is searched with five-fold city-group CV on training data only.
- Candidate comparison is capped at 10,000 rows; tuning is capped at 4,000 rows. The selected pipeline is refitted on all training rows.
- Classification selection uses mean CV F1 with ROC AUC as tie-break; regression uses lowest mean CV MAE.
- Training fold scores and train/test gaps help flag overfitting. Segment metrics describe the final city holdout.
- Target-generating rules are shown separately from fitted models. Results evaluate formula-generated proxies, not real investment outcomes.

## 1. Candidate model roster

| Task | Candidates |
|---|---|
| Classification | Most-frequent dummy, logistic regression, decision tree, random forest, extra trees, gradient boosting |
| Regression | Mean dummy, linear regression, ridge, decision tree, random forest, extra trees, gradient boosting |

The dummy estimators provide simple baselines. Linear and tree ensembles compare different model assumptions. A model is selected only from its grouped cross-validation results; the held-out cities are reserved for the final report.


Run this cell to compare model families, tune the CV-selected family, refit it on the complete training partition, and score it once on the untouched city holdout. The script writes comparison, tuning, segment, quality, and run-manifest outputs under artifacts/.

In [ ]:
runpy.run_path(str(PROJECT_ROOT / "scripts" / "train.py"), run_name="__main__")
print("Training and evaluation run finished.")


## 3. Classification - cross-validation results


In [ ]:
classification_cv = pd.read_csv(ARTIFACT_DIR / "classification_cv_results.csv")
classification_cv = classification_cv.sort_values("cv_f1_mean", ascending=False).reset_index(drop=True)
display(classification_cv)


### Classification CV comparison

Bars show mean F1 across the five city-group folds; error bars show fold-to-fold standard deviation. Higher F1 is better. Compare the selected candidate with the dummy baseline before interpreting the score.


In [ ]:
plot_data = classification_cv.sort_values("cv_f1_mean", ascending=True)
fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = ["#1B9E77" if name == classification_cv.iloc[0]["model"] else "#4C78A8" for name in plot_data["model"]]
ax.barh(
    plot_data["model"], plot_data["cv_f1_mean"],
    xerr=plot_data["cv_f1_std"], color=bar_colors,
    edgecolor="white", linewidth=1, capsize=4,
)
ax.set(title="Classifier selection: five-fold CV F1 by held-out city groups", xlabel="Mean CV F1 (higher is better)", ylabel="Candidate model")
ax.set_xlim(0, 1.08)
ax.grid(axis="x", alpha=0.25)
for y, value in enumerate(plot_data["cv_f1_mean"]):
    ax.text(value + 0.015, y, f"{value:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()


### Classification - untouched city holdout


In [ ]:
classification_holdout = pd.read_csv(ARTIFACT_DIR / "classification_holdout_results.csv")
classification_holdout = classification_holdout.sort_values("test_f1", ascending=False).reset_index(drop=True)
display(classification_holdout)


### Classification - rule reference

This is the explicit rule that generated the target, not a fitted model and not a CV result. Its purpose is to show how easy the proxy is to reconstruct.


In [ ]:
classification_reference = pd.read_csv(ARTIFACT_DIR / "classification_reference_results.csv")
display(classification_reference)


## 4. Regression - cross-validation results


In [ ]:
regression_cv = pd.read_csv(ARTIFACT_DIR / "regression_cv_results.csv")
regression_cv = regression_cv.sort_values("cv_mae_mean", ascending=True).reset_index(drop=True)
display(regression_cv)


### Regression CV comparison

Bars show mean MAE across the five city-group folds; error bars show fold-to-fold standard deviation. Lower MAE is better. Values are in lakhs and measure error against the formula-generated five-year price proxy.


In [ ]:
plot_data = regression_cv.sort_values("cv_mae_mean", ascending=False)
fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = ["#1B9E77" if name == regression_cv.iloc[0]["model"] else "#E07A5F" for name in plot_data["model"]]
ax.barh(
    plot_data["model"], plot_data["cv_mae_mean"],
    xerr=plot_data["cv_mae_std"], color=bar_colors,
    edgecolor="white", linewidth=1, capsize=4,
)
ax.set(title="Regressor selection: five-fold CV MAE by held-out city groups", xlabel="Mean CV MAE (lakhs; lower is better)", ylabel="Candidate model")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


### Regression - untouched city holdout


In [ ]:
regression_holdout = pd.read_csv(ARTIFACT_DIR / "regression_holdout_results.csv")
regression_holdout = regression_holdout.sort_values("test_mae_lakhs", ascending=True).reset_index(drop=True)
display(regression_holdout)


### Regression - formula reference

The target is generated by compounding today's asking price at the configured rate. This known formula is a reference, not a fitted model and not a CV-ranked candidate.


In [ ]:
regression_reference = pd.read_csv(ARTIFACT_DIR / "regression_reference_results.csv")
display(regression_reference)


## 5. Selected model and final metrics


In [ ]:
with (ARTIFACT_DIR / "metrics.json").open(encoding="utf-8") as file:
    metrics = json.load(file)

selection_summary = pd.DataFrame([
    {
        "task": "Classification",
        "selected_model": metrics["classification"]["selected_model"],
        "selection_rule": metrics["classification"]["selection_metric"],
        "tuned_parameters": metrics["classification"]["tuned_parameters"],
        "tuning_cv_f1": metrics["classification"]["tuning_best_cv_f1"],
        "CV strategy": metrics["classification"]["cv_strategy"],
        "holdout strategy": metrics["classification"]["test_strategy"],
        **metrics["classification"]["holdout"],
    },
    {
        "task": "Regression",
        "selected_model": metrics["regression"]["selected_model"],
        "selection_rule": metrics["regression"]["selection_metric"],
        "tuned_parameters": metrics["regression"]["tuned_parameters"],
        "tuning_cv_mae_lakhs": metrics["regression"]["tuning_best_cv_mae_lakhs"],
        "CV strategy": metrics["regression"]["cv_strategy"],
        "holdout strategy": metrics["regression"]["test_strategy"],
        **metrics["regression"]["holdout"],
    },
])
display(selection_summary)


## 6. Findings and interpretation

Use the CV comparison to choose a candidate, then use the held-out city results to assess transfer to locations the candidate did not see during training. A large gap between CV and holdout performance is a signal that performance does not generalize consistently across cities.

The rule-reference rows must not be presented as model performance. Even strong scores for the selected models mean only that a model approximates the engineered label or assumed price formula. For a real-estate investment use case, replace both targets with observed investor returns and time-stamped resale transactions, then validate on a later time period and locality-level holdouts.


## 7. Hyperparameter search results
These tables show the bounded grouped-CV search for the model family selected in the candidate comparison. Scores are training-CV selection evidence; use the untouched city holdout for final comparison.

In [ ]:
classification_tuning = pd.read_csv(ARTIFACT_DIR / 'classification_tuning_results.csv')
display(classification_tuning[['params', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

regression_tuning = pd.read_csv(ARTIFACT_DIR / 'regression_tuning_results.csv')
regression_tuning['mean_train_mae_lakhs'] = -regression_tuning['mean_train_score']
regression_tuning['mean_cv_mae_lakhs'] = -regression_tuning['mean_test_score']
display(regression_tuning[['params', 'mean_train_mae_lakhs', 'mean_cv_mae_lakhs', 'std_test_score', 'rank_test_score']].head(10))

## 8. Held-out performance by city and property type
Small segment counts make individual scores noisy. Compare row counts alongside metrics; these segment scores describe proxy reconstruction on cities withheld from training.

In [ ]:
classification_segments = pd.read_csv(ARTIFACT_DIR / 'classification_segment_metrics.csv')
regression_segments = pd.read_csv(ARTIFACT_DIR / 'regression_segment_metrics.csv')
display(classification_segments.sort_values(['segment', 'rows'], ascending=[True, False]))
display(regression_segments.sort_values(['segment', 'rows'], ascending=[True, False]))

## 9. Data quality and run record
The manifest makes the run reproducible against the same local input and assumptions. Missing build years are excluded only from classification because the label is derived from age; valid price rows remain in regression.

In [ ]:
with (ARTIFACT_DIR / 'run_manifest.json').open(encoding='utf-8') as file:
    manifest = json.load(file)
display(pd.Series({
    'input_file': manifest['source_file'],
    'input_sha256': manifest['source_sha256'],
    'cleaned_rows': manifest['cleaned_rows'],
    'classification_rows': manifest['classification_rows_with_known_year_built'],
    'regression_rows': manifest['regression_rows'],
    'python': manifest['python_version'],
    'package_versions': manifest['package_versions'],
}).to_frame('value'))
display(pd.Series(manifest['cleaning_summary']['missing_values_by_column'], name='missing rows').sort_values(ascending=False).head(12))